# Logistic Regression Exploration

This notebook walks through the logistic regression implementation step by step, visualizing the math at each stage.

In [ ]:
import sys

sys.path.insert(0, "src")

import numpy as np
import matplotlib.pyplot as plt

from from_zero.logistic import (
    sigmoid,
    binary_cross_entropy_with_logits,
    compute_gradients,
    check_gradients,
    LogisticRegression,
)
from from_zero.core.datasets import prepare_classification
from from_zero.core.plotting import plot_sigmoid, plot_decision_boundary

## 1. Sigmoid Function

The sigmoid maps any real number to (0, 1). Our implementation uses a piecewise formula for numerical stability.

In [ ]:
z = np.linspace(-10, 10, 200)
sig = sigmoid(z)

fig = plot_sigmoid()
plt.show()

print("sigmoid(-1000) = ", sigmoid(-1000.0))
print("sigmoid(1000) = ", sigmoid(1000.0))
print("sigmoid(0) = ", sigmoid(0.0))

## 2. Binary Cross-Entropy from Logits

The stable formulation uses `softplus(z) - y*z` instead of `log(sigmoid(z))`.

In [ ]:
# Perfect predictions
y = np.array([0, 1])
logits = np.array([-1000.0, 1000.0])
loss = binary_cross_entropy_with_logits(y, logits)
print("Perfect predictions loss:", loss)

# Wrong predictions
logits_wrong = np.array([1000.0, -1000.0])
loss_wrong = binary_cross_entropy_with_logits(y, logits_wrong)
print("Wrong predictions loss:", loss_wrong)

# Uncertain predictions
logits_uncertain = np.array([0.0, 0.0])
loss_uncertain = binary_cross_entropy_with_logits(y, logits_uncertain)
print("Uncertain predictions loss:", loss_uncertain)

## 3. Gradient Check

Compare analytical gradients with central finite differences.

In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(size=(30, 4))
y = (X[:, 0] - X[:, 1] + 0.2 * rng.normal(size=30) > 0).astype(int)
weights = rng.normal(size=4) * 0.3
bias = 0.1

result = check_gradients(X, y, weights, bias, regularization=0.01)
print("Passed:", result["passed"])
print("Max weight error:", result["max_abs_weight_error"])
print("Bias error:", result["abs_bias_error"])
print("Analytic weights:", result["analytic_weights"])
print("Numeric weights:", result["numeric_weights"])

## 4. Train on Linearly Separable Data

In [ ]:
prepared = prepare_classification(
    name="linearly_separable",
    n_samples=800,
    noise=0.22,
    test_fraction=0.2,
    val_fraction=0.2,
    seed=42,
)
split = prepared.split
print(f"Train: {split.X_train.shape}, Val: {split.X_val.shape}, Test: {split.X_test.shape}")

In [ ]:
model = LogisticRegression(
    learning_rate=0.08,
    epochs=400,
    batch_size=64,
    regularization=0.001,
    early_stopping_patience=80,
    seed=42,
)
model.fit(split.X_train, split.y_train, X_val=split.X_val, y_val=split.y_val)

print(f"Epochs run: {model.n_iter_}")
print(f"Best epoch: {model.best_epoch_}")
print(f"Converged: {model.converged_}")
print(f"Test accuracy: {model.score(split.X_test, split.y_test):.4f}")

## 5. Decision Boundary Visualization

In [ ]:
fig = plot_decision_boundary(
    model, split.X_train, split.y_train, title="Training set (standardized)"
)
plt.show()

fig = plot_decision_boundary(model, split.X_test, split.y_test, title="Test set (standardized)")
plt.show()

## 6. Nonlinear Limitation: Moons Dataset

A linear decision boundary cannot separate interleaving moons.

In [ ]:
prepared_moons = prepare_classification(
    name="moons",
    n_samples=600,
    noise=0.15,
    test_fraction=0.2,
    val_fraction=0.2,
    seed=42,
)
split_m = prepared_moons.split

model_m = LogisticRegression(
    learning_rate=0.1,
    epochs=500,
    batch_size=64,
    regularization=0.001,
    seed=42,
)
model_m.fit(split_m.X_train, split_m.y_train, X_val=split_m.X_val, y_val=split_m.y_val)

print(f"Test accuracy on moons: {model_m.score(split_m.X_test, split_m.y_test):.4f}")

fig = plot_decision_boundary(
    model_m, split_m.X_test, split_m.y_test, title="Moons test set — linear boundary limitation"
)
plt.show()